# 🔬 Test du modèle Conformer (.npy) sur les échantillons CSV bruités
## Évaluation de la généralisation : synthétique → réel
### CMKL University · Stage 2026

---

**Objectif** : tester le modèle Conformer PatchTST entraîné sur les données
synthétiques `.npy` de Dr. Sarun directement sur les vrais spectres bruités CSV,
sans réentraînement — pour évaluer la **généralisation domaine synthétique → réel**.

**Ce que ça teste** : la case *"Test on actual filter-interfered spectra"* du schéma de Dr. Sarun.

**Défis techniques** :
- Grille spectrale différente : CSV (1727 pts) → interpolation vers grille .npy (6700 pts)
- Labels CSV hétérogènes → nettoyage et alignement avec `ASSUMED_CLASSES`
- Bruit CSV ≠ bruit synthétique .npy (bruit réel vs interférence filtre simulée)

**Modèle chargé** : Conformer PatchTST (P=320, S=256, n=5, d=256, h=16, beta=0.5)
→ 95.25% sur le test officiel .npy (-30dB)


---
## ⚙️ Section 0 — Imports & Configuration


In [ ]:
import os, glob, re, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HOME   = os.path.expanduser('~')
print(f'Device : {DEVICE}')
print(f'HOME   : {HOME}')

In [ ]:
# ── Grille officielle .npy (sur laquelle le modèle a été entraîné) ────────
WN_GRID_NPY = np.arange(650, 4000, 0.5)   # 6700 points
L_NPY       = len(WN_GRID_NPY)
print(f'Grille .npy : {L_NPY} points (650-4000 cm⁻¹, pas 0.5)')

# ── Classes dans l'ordre alphabétique confirmé ────────────────────────────
ASSUMED_CLASSES = [
    'ABS', 'ACRYLIC', 'CELLULOSE', 'CHITOSAN', 'ENR', 'EPDM', 'EVA', 'HDPE',
    'LDPE', 'NYLON', 'PBAT', 'PBS', 'PC', 'PEEK', 'PEI', 'PET',
    'PF THERMOPLASTIC', 'PF THERMOSET', 'PHB', 'PLA', 'PMMA', 'POM', 'PP',
    'PS', 'PTFE', 'PU', 'PVA', 'PVC', 'PVDF', 'SAN',
]
N_CLASSES = len(ASSUMED_CLASSES)

le = LabelEncoder()
le.fit(ASSUMED_CLASSES)
print(f'Classes : {N_CLASSES}')

---
## 🏛️ Section 1 — Architecture du modèle Conformer

Copie exacte de l'architecture finale (identique au notebook d'entraînement).


In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)
    def forward(self, z):
        scores  = self.attn(z)
        weights = F.softmax(scores, dim=1)
        return (weights * z).sum(dim=1)


class PatchEmbedding(nn.Module):
    def __init__(self, L, patch_size, stride, d_model):
        super().__init__()
        self.P = patch_size; self.S = stride; self.D = d_model
        self.N = (L - patch_size) // stride + 2
        self.patch_proj = nn.Linear(patch_size, d_model)
        self.pos_embed  = nn.Embedding(self.N, d_model)
        self.dropout    = nn.Dropout(0.1)

    def get_raw_patches(self, x):
        B   = x.shape[0]
        pad = x[:, -1:].expand(B, self.S)
        x_pad = torch.cat([x, pad], dim=1)
        return x_pad.unfold(1, self.P, self.S)

    def forward(self, x):
        patches  = self.get_raw_patches(x)
        content  = self.patch_proj(patches)
        pos_vecs = self.pos_embed(torch.arange(self.N, device=x.device))
        return self.dropout(content + pos_vecs)


class ConformerFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.W1   = nn.Linear(d_model, d_ff)
        self.V    = nn.Linear(d_model, d_ff)
        self.W2   = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm(x)
        return self.drop(self.W2(F.silu(self.W1(x)) * self.V(x)))


class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pw1  = nn.Conv1d(d_model, 2*d_model, 1)
        self.glu  = nn.GLU(dim=1)
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size,
                               padding=kernel_size//2, groups=d_model)
        self.bn   = nn.BatchNorm1d(d_model)
        self.act  = nn.SiLU()
        self.pw2  = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        r = x
        x = self.norm(x).transpose(1,2)
        x = self.glu(self.pw1(x))
        x = self.act(self.bn(self.dw(x)))
        x = self.drop(self.pw2(x)).transpose(1,2)
        return r + x


class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, kernel_size=31):
        super().__init__()
        self.ffn1      = ConformerFFN(d_model, d_ff, dropout)
        self.attn      = nn.MultiheadAttention(d_model, n_heads,
                                                dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.conv      = ConformerConvModule(d_model, kernel_size, dropout)
        self.ffn2      = ConformerFFN(d_model, d_ff, dropout)
        self.norm      = nn.LayerNorm(d_model)
        self.drop      = nn.Dropout(dropout)
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        xn = self.attn_norm(x)
        x  = x + self.drop(self.attn(xn, xn, xn)[0])
        x  = self.conv(x)
        x  = x + 0.5 * self.ffn2(x)
        return self.norm(x)


class TransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm(x)


class ClassificationHead(nn.Module):
    def __init__(self, d_model, n_classes, dropout=0.1, hidden_dim=None):
        super().__init__()
        if hidden_dim is None: hidden_dim = d_model // 2
        self.attn_pool = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, n_classes),
        )
    def forward(self, z):
        w = F.softmax(self.attn_pool(z), dim=1)
        return self.head((w * z).sum(dim=1))


class DenoisingHead(nn.Module):
    def __init__(self, d_model, patch_size, n_patches, stride, spectrum_length):
        super().__init__()
        self.P = patch_size; self.S = stride
        self.N = n_patches;  self.L = spectrum_length
        self.proj = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Linear(d_model, patch_size))
    def forward(self, z):
        B = z.shape[0]
        pr = self.proj(z)
        out = torch.zeros(B, self.L+self.S, device=z.device)
        cnt = torch.zeros(self.L+self.S, device=z.device)
        for k in range(self.N):
            s = k * self.S
            out[:, s:s+self.P] += pr[:, k, :]
            cnt[s:s+self.P]    += 1
        return (out / cnt.clamp(min=1))[:, :self.L]


class PatchTSTMultiTask(nn.Module):
    def __init__(self, patch_embed, backbone, class_head, denoise_head,
                 alpha=1.0, beta=0.5):
        super().__init__()
        self.patch_embed  = patch_embed
        self.backbone     = backbone
        self.class_head   = class_head
        self.denoise_head = denoise_head
        self.alpha = alpha; self.beta = beta

    def encode(self, x):
        return self.backbone(self.patch_embed(x))

    def classify(self, x):
        return self.class_head(self.encode(x))

    def forward(self, x, y=None, clean_target=None):
        z        = self.encode(x)
        logits   = self.class_head(z)
        denoised = self.denoise_head(z)
        loss = None
        if y is not None and clean_target is not None:
            loss = self.alpha * F.cross_entropy(logits, y, label_smoothing=0.1) \
                 + self.beta  * F.mse_loss(denoised, clean_target)
        return logits, denoised, loss

print('✓ Architecture Conformer définie')

---
## 📦 Section 2 — Chargement du modèle entraîné


In [ ]:
# ── Config optimale finale (identique au run 95.25%) ─────────────────────
CFG = {
    'L'          : 6700,
    'patch_size' : 320,
    'stride'     : 256,
    'd_model'    : 256,
    'n_heads'    : 16,
    'n_layers'   : 5,
    'd_ff'       : 512,
    'dropout'    : 0.1,
    'beta'       : 0.5,
    'N_CLASSES'  : 30,
}

N_PATCHES = (CFG['L'] - CFG['patch_size']) // CFG['stride'] + 2
print(f'N_PATCHES = {N_PATCHES}')

# ── Instanciation ─────────────────────────────────────────────────────────
patch_embed  = PatchEmbedding(CFG['L'], CFG['patch_size'], CFG['stride'], CFG['d_model']).to(DEVICE)
backbone     = TransformerBackbone(CFG['d_model'], CFG['n_heads'], CFG['n_layers'],
                                    CFG['d_ff'], CFG['dropout']).to(DEVICE)
class_head   = ClassificationHead(CFG['d_model'], CFG['N_CLASSES'], dropout=0.1).to(DEVICE)
denoise_head = DenoisingHead(CFG['d_model'], CFG['patch_size'],
                              N_PATCHES, CFG['stride'], CFG['L']).to(DEVICE)

model = PatchTSTMultiTask(patch_embed, backbone, class_head, denoise_head,
                           alpha=1.0, beta=CFG['beta']).to(DEVICE)

# ── Charger les poids sauvegardés ─────────────────────────────────────────
MODEL_PATH = os.path.join(HOME, 'models', 'final_model_v2.pth')
ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f'✓ Modèle chargé depuis {MODEL_PATH}')
print(f'  Val Acc sur .npy : {ckpt["val_acc"]:.2%}')

---
## 📊 Section 3 — Chargement des données CSV bruités

Pipeline de lecture CSV identique au notebook 3-phases, avec en plus
**interpolation vers la grille 6700 points** du modèle .npy.


In [ ]:
# ── Chemins du dataset CSV sur Glider ────────────────────────────────────
# ⚠️ À ADAPTER selon où tu as dézippé le fichier
CSV_ROOT = os.path.join(HOME, 'data', 'csv-ftir-dataset')  # adapter si besoin

PATHS_NOISY = {
    '2023_noisy_50'  : os.path.join(CSV_ROOT, '2023 Dataset - 22 MP Types with 10 Clean and 60 Noisy',
                                     '2. Noisy Spectra - 60 Each', '22 Types - 50 Each'),
    '2023_noisy_10'  : os.path.join(CSV_ROOT, '2023 Dataset - 22 MP Types with 10 Clean and 60 Noisy',
                                     '2. Noisy Spectra - 60 Each', '22 Types - 10 Each'),
    '2025_noisy_new' : os.path.join(CSV_ROOT, '2025 Dataset 2 - New 9 MP Types - 50 Clean and 100 Noisy',
                                     '2. Noisy - 100 Each'),
}

for name, path in PATHS_NOISY.items():
    status = '✓' if os.path.exists(path) else '✗ INTROUVABLE'
    n = len(glob.glob(os.path.join(path, '**/*.csv'), recursive=True)) if os.path.exists(path) else 0
    print(f'  {status}  {name:20s}  {n} CSV')

In [ ]:
EXCLUDE_FILES = {'ref.csv', 'reference.csv', 'background.csv', 'bg.csv'}

def read_csv_multispectra(filepath, sep=','):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        start_idx = 0
        for i, line in enumerate(lines):
            parts = line.strip().split(sep)
            if len(parts) >= 2:
                try:
                    float(parts[0].replace(',', '.'))
                    start_idx = i; break
                except ValueError:
                    continue
        valid = ''.join(lines[start_idx:])
        headers = lines[start_idx-1].strip().split(sep) if start_idx > 0 else []
        try:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal=',')
        except Exception:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal='.')

        cols = []
        for ci, cn in enumerate(df.columns):
            h = headers[ci].upper() if ci < len(headers) else ''
            v = str(df[cn].iloc[0]).upper()
            if any(k in h for k in ['AIR','BACKGROUND','BG']): continue
            if any(k in v for k in ['AIR','BACKGROUND','BG']): continue
            cols.append(cn)

        df = df[cols].apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=[df.columns[0]])
        if len(df) < 100: return None

        wn    = df.iloc[:, 0].values.astype(float)
        order = np.argsort(wn); wn = wn[order]
        spectra = []
        for c in range(1, df.shape[1]):
            ab = df.iloc[order, c].values.astype(float)
            if np.isnan(ab).all() or ab.std() < 1e-10: continue
            nans = np.isnan(ab)
            if nans.any():
                ab[nans] = np.interp(np.where(nans)[0], np.where(~nans)[0], ab[~nans])
            spectra.append(ab.astype(np.float32))
        return (wn, spectra) if spectra else None
    except Exception as e:
        return None


def extract_label_csv(filepath):
    """Extrait le label depuis le nom de fichier CSV et l'aligne avec ASSUMED_CLASSES."""
    name = Path(filepath).stem.upper()
    # Supprimer les suffixes parasites
    for pattern in ['_SD_', '_RM_', '_NOISY', '_CLEAN', 'PARTICLE', '-NOISY',
                    '-CLEAN', '_50', '_60', '_40', '_100', '_10', '_30',
                    ' SPECTRUMS', ' SPECTUMS', 'ADD_40', '-ADD_40']:
        name = name.replace(pattern, ' ')
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'\bNEW\b|\bJAN\b|\bX\b', '', name)
    name = ' '.join(name.replace('_', ' ').replace('-', ' ').split())

    # Mapping des noms CSV vers ASSUMED_CLASSES
    MAPPING = {
        'ACRYLIC'          : 'ACRYLIC',
        'CELLULOSE'        : 'CELLULOSE',
        'CHITOSAN'         : 'CHITOSAN',
        'NYLON PARTICLE'   : 'NYLON',
        'PTEE'             : 'PTFE',   # faute de frappe historique
        'PTFE'             : 'PTFE',
        'PF THERMOPLASTIC' : 'PF THERMOPLASTIC',
        'PF THERMOSET'     : 'PF THERMOSET',
        'PF THERMOPLASTIC CLEAN' : 'PF THERMOPLASTIC',
        'PF THERMOSET CLEAN'     : 'PF THERMOSET',
        'PHB'              : 'PHB',
        'PVDF'             : 'PVDF',
        'ABS'              : 'ABS',
        'SAN'              : 'SAN',
        'EVA'              : 'EVA',
    }
    if name in MAPPING:
        return MAPPING[name]
    # Vérification directe
    if name in ASSUMED_CLASSES:
        return name
    # Recherche partielle
    for cls in ASSUMED_CLASSES:
        if cls in name or name in cls:
            return cls
    return None   # pas trouvé


print('✓ Fonctions de lecture définies')

In [ ]:
print('Chargement des spectres CSV bruités...')
records = []
labels_not_found = Counter()

for src_name, folder in PATHS_NOISY.items():
    if not os.path.exists(folder):
        print(f'  ✗ {src_name} introuvable'); continue

    files = glob.glob(os.path.join(folder, '**/*.csv'), recursive=True)
    n_ok, n_skip = 0, 0

    for fp in files:
        if Path(fp).name.lower() in EXCLUDE_FILES: continue

        label = extract_label_csv(fp)
        if label is None:
            labels_not_found[Path(fp).stem] += 1
            n_skip += 1
            continue

        result = read_csv_multispectra(fp)
        if result is None: n_skip += 1; continue

        wn, spectra_list = result
        for sp in spectra_list:
            # ── Interpolation vers la grille .npy (6700 pts) ──────────────
            sp_interp = np.interp(WN_GRID_NPY, wn, sp).astype(np.float32)
            records.append({'label': label, 'source': src_name, 'spectrum': sp_interp})
            n_ok += 1

    print(f'  ✓ {src_name:20s} : {n_ok:4d} spectres, {n_skip} ignorés')

df_csv = pd.DataFrame(records)
df_csv['label_enc'] = le.transform(df_csv['label'])

print(f'\n  Total spectres CSV bruités : {len(df_csv)}')
print(f'  Classes présentes          : {df_csv.label.nunique()}')
print(f'  Distribution :')
print(df_csv.label.value_counts().to_string())

if labels_not_found:
    print(f'\n  ⚠️ Labels non reconnus : {dict(labels_not_found)}')

---
## 🔍 Section 4 — Inférence sur les spectres CSV bruités


In [ ]:
class CSVInferenceDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.spectra = np.stack(df['spectrum'].values)
        self.labels  = df['label_enc'].values.astype(int)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.spectra[idx], dtype=torch.float32)
        x = (x - x.mean()) / (x.std() + 1e-8)   # instance norm
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


csv_dataset = CSVInferenceDataset(df_csv)
csv_loader  = DataLoader(csv_dataset, batch_size=64, shuffle=False, num_workers=0)

# ── Inférence ─────────────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []
all_probs = []

with torch.no_grad():
    for x, y in csv_loader:
        x = x.to(DEVICE)
        logits = model.classify(x)
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(y.numpy())
        all_probs.extend(probs)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

final_acc = accuracy_score(all_labels, all_preds)

print('═'*60)
print(f'  Test sur spectres CSV bruités (domaine réel)')
print(f'  Accuracy : {final_acc:.2%}')
print(f'  (Référence : 95.25% sur test officiel .npy synthétique)')
print('═'*60)

In [ ]:
print(classification_report(all_labels, all_preds,
                             target_names=le.classes_, zero_division=0))

In [ ]:
# ── Matrice de confusion ──────────────────────────────────────────────────
classes_present = sorted(df_csv['label_enc'].unique())
classes_names   = [le.classes_[i] for i in classes_present]

cm      = confusion_matrix(all_labels, all_preds, labels=classes_present)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, data, fmt, title in zip(axes, [cm, cm_norm], ['d', '.2f'],
    ['Matrice de confusion (counts)', 'Matrice de confusion (normalisée)']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=classes_names, yticklabels=classes_names,
                ax=ax, annot_kws={'size':7})
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle(f'Conformer .npy → CSV bruités | Acc : {final_acc:.2%}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(HOME, 'confusion_csv_inference.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Analyse de la confiance ───────────────────────────────────────────────
max_probs = all_probs.max(axis=1)
correct   = (all_preds == all_labels)

print(f'=== Analyse de confiance ===')
print(f'  Confiance moyenne (toutes prédictions)   : {max_probs.mean():.1%}')
print(f'  Confiance moyenne (prédictions correctes) : {max_probs[correct].mean():.1%}')
print(f'  Confiance moyenne (prédictions erronées)  : {max_probs[~correct].mean():.1%}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(max_probs[correct]*100,  bins=30, alpha=0.7, color='steelblue',
        label=f'Correct ({correct.sum()}) μ={max_probs[correct].mean():.0%}')
ax.hist(max_probs[~correct]*100, bins=30, alpha=0.7, color='coral',
        label=f'Erreur ({(~correct).sum()}) μ={max_probs[~correct].mean():.0%}')
ax.set_xlabel('Confiance max (%)'); ax.set_ylabel('Fréquence')
ax.set_title('Distribution de confiance — CSV bruités', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Comparaison synthétique vs réel ──────────────────────────────────────
print('═'*65)
print('  BILAN — Généralisation synthétique → réel')
print('═'*65)
print(f'  Entraînement sur .npy synthétique (-30dB) : 95.25%')
print(f'  Test sur CSV bruités réels                : {final_acc:.2%}')
print(f'  Écart (domain gap)                        : {0.9525 - final_acc:+.2%}')
print()

# Accuracy par classe
print('  Accuracy par classe (CSV réel) :')
per_class = {}
for cls_idx in classes_present:
    mask = (all_labels == cls_idx)
    acc_cls = (all_preds[mask] == cls_idx).mean()
    per_class[le.classes_[cls_idx]] = acc_cls

for cls, acc_cls in sorted(per_class.items(), key=lambda x: x[1]):
    bar = '█' * int(acc_cls * 20)
    print(f'  {cls:20s} : {acc_cls:6.1%} {bar}')